# Phase A: Validate Data Access

Lightweight checks that all required data sources, packages, and write
permissions are available. 


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from gcsfs import GCSFileSystem
from cloudpathlib import AnyPath

In [3]:
#!pip install cloudpathlib
#!pip install python-CIAM

In [4]:

from config import (
    PATH_PARAMS,
    PATH_SLIIDERS,
    PATH_SLIIDERS_SEG,
    PATH_SLR_INEQUALITY,
    PATH_REFA_INEQUALITY,
    PATHS_SURGE_LOOKUP,
    PATH_OUTPUT_TMP,
    PATH_OUTPUT_FINAL,
    DIR_SCRATCH,
    DIR_SLR_AR6_RAW,
    DIR_SLR_AR6_GRIDDED_PUBLIC,
    PATH_VLM_REQUESTER_PAYS,
    TLIM_SCENARIOS,
    WORKFLOWS,
    STORAGE_OPTIONS,
    save_zarr,
    open_dataset,
)

print("Config loaded.")

Config loaded.


## Check 1: params.json

In [5]:
try:
    params = pd.read_json(PATH_PARAMS)['values']
    assert hasattr(params, 'dr'), "Missing 'dr'"
    assert hasattr(params, 'movefactor'), "Missing 'movefactor'"
    assert hasattr(params, 'at_start'), "Missing 'at_start'"
    print(f"  params.json -- dr={params.dr}, movefactor={params.movefactor}, model_start={params.model_start}")
except Exception as e:
    print(f"  params.json -- FAIL: {e}")
    print(f"   Expected at: {PATH_PARAMS}")

  params.json -- dr=0.02, movefactor=7.6, model_start=2005


## Check 2: pyCIAM package

In [8]:
try:
    from pyCIAM.run import calc_all_cases, get_refA, optimize_case
    from pyCIAM.constants import CASES, COSTTYPES
    from pyCIAM.io import create_template_dataarray, load_ciam_inputs
    from pyCIAM.utils import collapse_econ_inputs_to_seg, subset_econ_inputs
    print(f"  pyCIAM  cases={CASES}, costtypes={len(COSTTYPES)}")
except Exception as e:
    print(f"  pyCIAM  FAIL: {e}")
    print("   Try: pip install python-CIAM")

  pyCIAM  cases=['noAdaptation', 'protect10', 'protect100', 'protect1000', 'protect10000', 'retreat1', 'retreat10', 'retreat100', 'retreat1000', 'retreat10000', 'optimalfixed'], costtypes=6


## Check 3: SLIIDERS

In [9]:
try:
    ds = xr.open_zarr(str(PATH_SLIIDERS), chunks=None)
    n_seg_ir = len(ds.seg_ir) if 'seg_ir' in ds.dims else '?'
    n_seg = len(ds.seg) if 'seg' in ds.dims else '?'
    print(f"  SLIIDERS  {n_seg_ir} seg_ir, {n_seg} seg, dims: {list(ds.dims)}")
    ds.close()
except Exception as e:
    print(f"  SLIIDERS  FAIL: {e}")
    print(f"   Path: {PATH_SLIIDERS}")

  SLIIDERS  19714 seg_ir, ? seg, dims: ['elev', 'bound', 'seg_ir', 'params', 'ssp', 'iam', 'year', 'country', 'return_period']


In [10]:
import importlib, pyCIAM.run
importlib.reload(pyCIAM.run)
import inspect
print(f"get_refA: {inspect.signature(pyCIAM.run.get_refA)}")
print(f"optimize_case: {inspect.signature(pyCIAM.run.optimize_case)}")

get_refA: (segs, econ_input_path, slr_input_path, params, surge_input_path=None, mc_dim='quantile', slr_site_id_dim='site_id', lsl_var='lsl_msl05', storage_options=None, quantile=0.5, eps=1, scen_mc_filter=None, diaz_inputs=False, output_path=None, **model_kwargs)
optimize_case: (selectors, *wait_futs, econ_input_path=None, output_path=None, seg_var='seg_adm', check=True, eps=1, storage_options=None)


## Check 4: Surge lookup tables

In [10]:
for name, path in PATHS_SURGE_LOOKUP.items():
    try:
        ds = xr.open_zarr(str(path), chunks=None)
        print(f"  Surge ({name})  dims: {list(ds.dims)}")
        ds.close()
    except Exception as e:
        print(f"  Surge ({name})  FAIL: {e}")
        print(f"   Path: {path}")

  Surge (seg)  dims: ['seg', 'lslr', 'rh_diff', 'costtype', 'adapttype']
  Surge (seg_ir)  dims: ['seg_ir', 'lslr', 'rh_diff', 'costtype', 'adapttype']


## Check 5: SLR data (for NB01)

In [12]:
# Local SLR (public bucket)
try:
    test_path = f"{DIR_SLR_AR6_GRIDDED_PUBLIC}/wf_1f/tlim2.0win0.25/total-workflow.zarr"
    ds = xr.open_zarr(test_path)
    n_samples = ds.dims.get('samples', '?')
    n_locations = ds.dims.get('locations', '?')
    print(f"  Local SLR (public)  {n_samples} samples, {n_locations} locations")
    ds.close()
except Exception as e:
    print(f"  Local SLR (public) FAIL: {e}")

  Local SLR (public)  20000 samples, ? locations


/tmp/ipykernel_2063/3671276567.py:5: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_samples = ds.dims.get('samples', '?')
<frozen _collections_abc>:807: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
/tmp/ipykernel_2063/3671276567.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_locations = ds.dims.get('locations', '?')


In [13]:
# Global SLR
try:
    gsl_path = DIR_SLR_AR6_RAW / 'wf_1f' / 'tlim2.0win0.25' / 'total-workflow.nc'
    ds = open_dataset(gsl_path)
    print(f"  Global SLR -- dims: {list(ds.dims)}")
    ds.close()
except Exception as e:
    print(f"  Global SLR -- FAIL: {e}")
    print(f"   Path: {gsl_path}")

  Global SLR -- dims: ['samples', 'years', 'locations']


getfattr: /gcs/impactlab-data/coastal/data/raw/slr/ar6/ar6/global/full_sample_workflows/wf_1f/tlim2.0win0.25/total-workflow.nc: Operation not supported


In [15]:
# VLM (requester-pays bucket)
try:
    fs = GCSFileSystem(requester_pays=True)
    mapping = fs.get_mapper(PATH_VLM_REQUESTER_PAYS)
    ds = xr.open_zarr(mapping)
    print(f"  VLM (requester-pays) dims: {list(ds.dims)}")
    ds.close()
except Exception as e:
    print(f"  VLM (requester-pays) FAIL: {e}")
    print("   This requires compute cluster with requester-pays access.")

  VLM (requester-pays) dims: ['samples', 'years', 'lat', 'lon']


## Check 6: Write access to scratch

In [17]:
try:
    test_ds = xr.Dataset({'x': xr.DataArray([1, 2, 3])})
    test_path = DIR_SCRATCH / 'validate-write-test.zarr'
    save_zarr(test_ds, test_path, mode='w')
    verify = xr.open_zarr(str(test_path))
    assert verify.x.values.tolist() == [1, 2, 3]
    print(f"  Write access scratch dir works: {DIR_SCRATCH}")
except Exception as e:
    print(f"  Write access FAIL: {e}")

  Write access scratch dir works: gs://impactlab-data-scratch/inequality-pyciam


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Check 7: Dask

In [21]:
try:
    from dask_gateway import Gateway
    gw = Gateway()
    clusters = gw.list_clusters()
    print(f"  Dask Gateway connected, {len(clusters)} existing clusters")
except Exception as e:
    print(f"  Dask Gateway FAIL: {e}")

  Dask Gateway connected, 0 existing clusters


## Check 8: Processed SLR (output of NB01)

This checks if NB01 has already been run. If not, you need to run it first.

In [23]:
try:
    ds = xr.open_zarr(str(PATH_SLR_INEQUALITY), chunks=None)
    print(f"  Processed SLR -- dims: {dict(ds.dims)}")
    print(f"   Scenarios: {ds.scenario.values}")
    print(f"   Samples: {len(ds.sample)}")
    print(f"   Sites: {len(ds.site_id)}")
    ds.close()
except Exception as e:
    print(f"  Processed SLR not found run 01_process_slr_inputs.ipynb first")
    print(f"   Path: {PATH_SLR_INEQUALITY}")

  Processed SLR not found run 01_process_slr_inputs.ipynb first
   Path: gs://impactlab-data-scratch/inequality-pyciam/ar6-tlim-slr-1000samples.zarr


---
## Summary

In [24]:
print()
print("=" * 60)
print("If all checks passed, proceed to:")
print("  1. 01_process_slr_inputs.ipynb (if Check 8 failed)")
print("  2. 00b_test_run_and_estimate_costs.ipynb")
print("  3. 02_run_pyciam_inequality.ipynb (production run)")
print("=" * 60)


If all checks passed, proceed to:
  1. 01_process_slr_inputs.ipynb (if Check 8 failed)
  2. 00b_test_run_and_estimate_costs.ipynb
  3. 02_run_pyciam_inequality.ipynb (production run)
